# Lab: register an agent in AWS Agent Registry and in Gemini Enterprise Agent Platform

In about 15 minutes you define a small agent with an A2A agent card, register it in both registries,
confirm that consumers can discover it, and delete everything again. Part A (AWS) and Part B
(Google Cloud) are independent; run either one alone if you only have one sandbox.

| | AWS Agent Registry | Gemini Enterprise Agent Platform |
|---|---|---|
| API | `agent-registry-control` to publish, `agent-registry` to discover (boto3) | `agentregistry.googleapis.com` (`google-cloud-agentregistry`) |
| You create | a **registry**, then one **record** per agent | one **Service** per agent; the registry derives a read-only **Agent** |
| Approval | `DRAFT` → submit → `APPROVED` (automatic or by a curator); only approved records are discoverable | none; listed and searchable right after creation |
| Discover | search, list, registry MCP endpoint | `agents.list`, `agents.search`, registry MCP server |

**Before you start**

* A sandbox AWS account and Google Cloud project. The last cell deletes what the lab creates.
* AWS credentials in your environment, and `gcloud auth application-default login` for Google Cloud.
* `pip install "boto3>=1.43.68" "google-cloud-agentregistry>=0.1.2" google-auth`
* AWS: permission for the `agent-registry:*` actions on the sandbox account, in a launch Region
  (`us-east-1`, `us-west-2`, `eu-west-1`, `ap-northeast-1`, `ap-southeast-2`).
  Google Cloud: `roles/agentregistry.admin` and `gcloud services enable agentregistry.googleapis.com`.

## Setup: settings and the agent card

Both registries describe an agent with an [A2A agent card](https://a2a-protocol.org/latest/specification/).
The card below is the whole "agent" for this lab. Set `AGENT_URL` to a real A2A endpoint if you have
one; the placeholder domain never resolves, which is fine for registration.

In [ ]:
import json
import os
import time
import uuid

AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
GCP_PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT", "")  # empty: the ADC project
GCP_LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")
AGENT_URL = os.environ.get("AGENT_URL", "")  # your agent's A2A endpoint, if you have one

RUN_ID = uuid.uuid4().hex[:6]
AGENT_NAME = f"lab-agent-{RUN_ID}"
AGENT_CARD = {
    "protocolVersion": "0.3.0",
    "name": AGENT_NAME,
    "description": "Demonstration agent from the agent registry lab. It answers short questions.",
    "url": AGENT_URL or f"https://lab.example.invalid/agents/{AGENT_NAME}",
    "version": "1.0.0",
    "capabilities": {"streaming": False},
    "defaultInputModes": ["text/plain"],
    "defaultOutputModes": ["text/plain"],
    "skills": [
        {"id": "lab-echo", "name": "Lab echo", "description": "Answers short questions.", "tags": ["lab"]}
    ],
}


def wait_for(describe, done, every=5, timeout=300):
    """Poll describe() until done(state) is true, printing the status whenever it changes."""
    deadline, last = time.monotonic() + timeout, None
    while True:
        state = describe()
        if state["status"] != last:
            last = state["status"]
            print("  status:", last)
        if done(state):
            return state
        if time.monotonic() > deadline:
            raise TimeoutError(state)
        time.sleep(every)


print("agent:", AGENT_NAME)

# Part A: AWS Agent Registry

## A1. Create a registry

A registry is an account- and Region-scoped catalog. Creation is asynchronous, so wait for `READY`.
`AWS_IAM` discovery means consumers call the data plane with their own IAM identity; `APPROVE_ALL`
auto-approves every submitted record, which keeps this lab short.

In [ ]:
import boto3

control = boto3.client("agent-registry-control", region_name=AWS_REGION)  # publish and curate
discovery = boto3.client("agent-registry", region_name=AWS_REGION)  # what consumers call

registry_arn = control.create_registry(
    name=f"lab-registry-{RUN_ID}",
    description="Agent registry lab",
    discoveryConfiguration={"authorizerType": "AWS_IAM"},
    approvalConfiguration={"autoApprovalRules": ["APPROVE_ALL"]},  # omit this line to require a curator
)["registryArn"]
registry_id = registry_arn.rsplit("/", 1)[-1]
print("registry:", registry_arn)

registry = wait_for(lambda: control.get_registry(registryId=registry_id), lambda r: r["status"] != "CREATING")
assert registry["status"] == "READY", registry.get("statusReason")

## A2. Register the agent and submit it for approval

An `AGENT` record stores the agent card as a JSON string (`dataSchemaVersion` `0.3` for A2A 0.3 cards).
The record is created asynchronously and lands in `DRAFT`. Submitting it moves it to `APPROVED`
here; without auto-approval it waits in `PENDING_APPROVAL` until a curator calls
`update_registry_record_status(..., status="APPROVED", statusReason=...)`.

In [ ]:
record_arn = control.create_registry_record(
    registryId=registry_id,
    name=AGENT_NAME,
    displayName="Lab agent",
    description=AGENT_CARD["description"],
    recordType="AGENT",
    descriptors={"a2aAgentCard": {"data": json.dumps(AGENT_CARD), "dataSchemaVersion": "0.3"}},
    recordVersion="1.0.0",
)["recordArn"]
print("record:", record_arn)

record = wait_for(
    lambda: control.get_registry_record(registryId=registry_id, recordId=record_arn),
    lambda r: r["status"] not in {"CREATING", "UPDATING"},
)
assert record["status"] == "DRAFT", record.get("statusReason")

submitted = control.submit_registry_record_for_approval(registryId=registry_id, recordId=record_arn)
print("after submission:", submitted["status"])

## A3. Discover it as a consumer

The `agent-registry` data plane returns approved records only. Listing is immediate; natural-language
search ranks by relevance. The same registry can be queried as an MCP server (the console shows
the endpoint; callers need `agent-registry:InvokeRegistryMcp`).

In [ ]:
listed = discovery.list_discoverable_registry_records(registryId=registry_id)["registryRecords"]
print("discoverable:", [(r["name"], r["recordType"], r["status"]) for r in listed])

hits = discovery.search_discoverable_registry_records(
    searchQuery="agent that answers short questions", registryIds=[registry_id], maxResults=5
)["registryRecords"]
for hit in hits:
    card = json.loads(hit["descriptors"]["a2aAgentCard"]["data"])
    print(f"search hit: {hit['name']} v{hit['recordVersion']} -> {card['url']}")

# Part B: Gemini Enterprise Agent Platform

## B1. Register the agent

In the Agent Registry API you write a **Service**; its `agent_spec` of type `A2A_AGENT_CARD` carries
the card inline. The call is a long-running operation. When it finishes, `registry_resource` names
the derived read-only **Agent**, whose display name, skills and protocols come from the card.
(Agents you deploy to Agent Runtime are registered this way automatically.)

In [ ]:
import google.auth
from google.cloud import agentregistry_v1 as ar

credentials, adc_project = google.auth.default()
GCP_PROJECT = GCP_PROJECT or adc_project
parent = f"projects/{GCP_PROJECT}/locations/{GCP_LOCATION}"
registry_client = ar.AgentRegistryClient(credentials=credentials)

service = ar.Service(
    display_name="Lab agent",
    description=AGENT_CARD["description"],
    agent_spec=ar.Service.AgentSpec(type_=ar.Service.AgentSpec.Type.A2A_AGENT_CARD, content=AGENT_CARD),
)
operation = registry_client.create_service(parent=parent, service_id=AGENT_NAME, service=service)
service = operation.result(timeout=300)
agent = registry_client.get_agent(name=service.registry_resource)
print("service:", service.name)
print("agent:  ", agent.name, "|", agent.display_name, "| skills:", [skill.id for skill in agent.skills])

## B2. Discover it

Consumers only read the derived resources: `list_agents` for everything in the location,
`search_agents` for field queries such as `skills.tags:lab` or `displayName:lab*`.

In [ ]:
for found in registry_client.search_agents(parent=parent, search_string="skills.tags:lab"):
    protocols = [protocol.type_.name for protocol in found.protocols]
    print(found.name, "|", found.display_name, "|", protocols)

# Cleanup

Records go before the registry on AWS; the Google service deletion removes the derived agent.

In [ ]:
from botocore.exceptions import ClientError

control.delete_registry_record(registryId=registry_id, recordId=record_arn)
for _ in range(60):  # the record disappears asynchronously; the registry cannot go before it
    try:
        control.get_registry_record(registryId=registry_id, recordId=record_arn)
    except ClientError as exc:
        if exc.response["Error"]["Code"] != "ResourceNotFoundException":
            raise
        break
    time.sleep(5)
control.delete_registry(registryId=registry_id)
print("AWS: registry and record deleted")

registry_client.delete_service(name=service.name).result(timeout=300)
print("Google Cloud: service and derived agent deleted")

# Where to go next

* **Point the card at a running agent.** On AWS, host an A2A server on
  [AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-a2a.html)
  and let the record sync its card from the endpoint (`descriptors.a2aAgentCard.source.fromUrl`).
  On Google Cloud, agents deployed to
  [Agent Runtime](https://docs.cloud.google.com/gemini-enterprise-agent-platform/agents) register automatically.
* **Govern instead of auto-approve.** Create the AWS registry without `approvalConfiguration`;
  submissions then raise EventBridge events and wait for `update_registry_record_status`
  ([record lifecycle](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/registry-record-lifecycle.html)).
  On Google Cloud, separate `roles/agentregistry.viewer` (consumers) from `roles/agentregistry.admin` (publishers).
* **Show it to end users.** Register the agent on a Gemini Enterprise app
  ([A2A agents](https://docs.cloud.google.com/gemini/enterprise/docs/register-and-manage-an-a2a-agent)).
* **Reconcile with ShadowScan.** The scanner observes deployed runtimes, not registry entries. Bind the
  runtime identifiers in a [sanctioned inventory](https://github.com/aisecnomad/Project-Nexus/blob/main/docs/inventory.md)
  so that a scan reports the agent as registered.